# Comparación de grafos compactos: K2-Tree vs Turán (PEMB) vs Lista de Adyacencia

Este notebook genera los gráficos de comparación entre las tres
representaciones evaluadas en el proyecto `Grafos-Planares-Compactos-EDC`:

- **K2-Tree**
- **Turán / PEMB** (extensión de Turán)
- **Lista de adyacencia** (representación clásica, usada como referencia)

A partir de los CSV de resultados se comparan:
- El **tamaño comprimido** (`size_in_megabytes`) de cada estructura, y el
  ahorro de espacio de las estructuras compactas respecto a la lista de
  adyacencia clásica.
- Los **tiempos de ejecución** de consultas de grado (1000 repeticiones) y
  de vecinos (500 repeticiones), incluyendo su variabilidad (`std_t`,
  `min_t`, `max_t`).
- La relación tamaño/tiempo entre las tres estructuras.


In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore")

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 11

## 1. Configuración

Ajustá estas variables según la ubicación real de tus archivos y el
contenido de la columna `explanation`.


In [ ]:
# Carpeta donde estan los CSV de resultados (relativa a la ubicacion del notebook).
RESULTS_DIR = Path("../results")

# Carga el csv que se le especifique, si se deja vacio (""), 
# el notebook carga y combina TODOS los .csv que
# encuentre en RESULTS_DIR.
CSV_FILENAME = ""

# Cantidad de repeticiones usadas en cada tipo de consulta (solo para
# mostrarlo en los titulos de los graficos de tiempo).
REPS_GRADO = 5000
REPS_VECINOS = 2500

# Unidad de las columnas de tiempo (avg_t, min_t, max_t, std_t).
# El benchmark del proyecto reporta en microsegundos.
UNIDAD_TIEMPO = "µs"

# Escalas logaritmicas: los tiempos y tamanos entre estructuras pueden
# diferir en varios ordenes de magnitud
# Con escala log las tres estructuras se ven bien en el mismo grafico.
ESCALA_LOG_TIEMPO = True
ESCALA_LOG_TAMANO = False

# Estructura que se usa como referencia/baseline para calcular el ahorro
# de espacio de las estructuras compactas (debe coincidir con una de las
# claves de PALABRAS_ESTRUCTURA de mas abajo).
BASELINE_ESTRUCTURA = "Lista de Adyacencia"

# Palabras clave para reconocer la ESTRUCTURA dentro de la columna
# 'explanation'. Agrega o modifica variantes segun como nombres tus
# experimentos (la comparacion se hace en minusculas, por substring).
PALABRAS_ESTRUCTURA = {
    "K2-Tree": ["k2tree", "k2-tree", "k2_tree", "k2 tree", "ktree", "k2"],
    "Turán (PEMB)": ["turan", "turán", "pemb"],
    "Lista de Adyacencia": ["ady_list", "adylist", "adyacencia", "adjacency", "adj_list"],
}

# Palabras clave para reconocer el TIPO DE CONSULTA dentro de 'explanation'.
PALABRAS_EXPERIMENTO = {
    "Grados": ["degree", "grado", "grados"],
    "Vecinos": ["neighbour", "neighbor", "vecino", "vecinos"],
}

# Paleta de colores fija por estructura, para que sea consistente en todos
# los graficos.
PALETA = {
    "K2-Tree": "#2E86AB",
    "Turán (PEMB)": "#E76F51",
    "Lista de Adyacencia": "#6A994E",
    "Otro": "#8D8D8D",
}

# Alias para limpiar el nombre de dataset que viene en 'label' (ej:
# 'planar_embedding1000000.pg' -> 'Planar-1M'). La clave es el prefijo en
# minusculas (sin numeros ni extension), el valor es como se muestra.
# Agrega mas entradas si usas otros datasets (WorldCities, Tiger-USA, etc).
ALIAS_PREFIJOS = {
    "planar_embedding": "Planar",
    "planarembedding": "Planar",
    "tiger_usa": "Tiger-USA",
    "tigerusa": "Tiger-USA",
    "worldcities": "WorldCities",
    "world_cities": "WorldCities",
}

# Tamano real (sin comprimir) del grafo de entrada en disco, en MB. La clave
# es el numero que aparece en 'label' (ej: 1000000 para planar_embedding1000000.pg).
# Se usa para expresar tamanos/ahorros respecto al grafo real en vez de solo
# respecto a otra estructura. Si un dataset no esta aca, se sigue mostrando
# con el formato anterior (ej "Planar-1M").
TAMANOS_REALES_MB = {
    1000000: 78.8,
    5000000: 455,
    10000000: 902,
}


## 2. Carga de datos

In [ ]:
def cargar_csvs(carpeta: Path, nombre_archivo: str) -> pd.DataFrame:
    carpeta = Path(carpeta)
    if nombre_archivo:
        rutas = [carpeta / nombre_archivo]
    else:
        rutas = sorted(carpeta.glob("*.csv"))

    if not rutas:
        raise FileNotFoundError(
            f"No se encontro ningun .csv en {carpeta.resolve()}. "
            "Revisa RESULTS_DIR y CSV_FILENAME en la celda de configuracion."
        )

    dfs = []
    for ruta in rutas:
        if not ruta.exists():
            print(f"Aviso: no existe {ruta}, se omite.")
            continue
        df_tmp = pd.read_csv(ruta)
        df_tmp["archivo_origen"] = ruta.name
        dfs.append(df_tmp)
        print(f"Cargado {ruta.name}: {len(df_tmp)} filas")

    return pd.concat(dfs, ignore_index=True)


df_raw = cargar_csvs(RESULTS_DIR, CSV_FILENAME)
df_raw.head()


## 3. Limpieza y extracción de columnas

Se generan `structure` (estructura detectada), `experiment_type` (tipo de
consulta) y `dataset` (nombre legible del grafo, ordenado numéricamente por
tamaño en vez de alfabéticamente).


In [ ]:
def detectar_categoria(texto: str, diccionario: dict) -> str:
    texto = str(texto).lower()
    for categoria, variantes in diccionario.items():
        for variante in variantes:
            if variante in texto:
                return categoria
    return "Otro"


def formatear_cantidad(n: int) -> str:
    if n >= 1_000_000:
        return f"{n / 1_000_000:g}M"
    if n >= 1_000:
        return f"{n / 1_000:g}K"
    return str(n)


def nombre_y_orden_dataset(label: str):
    """Devuelve (nombre_legible, clave_de_orden) a partir de 'label'."""
    base = re.sub(r"\.[A-Za-z0-9]+$", "", str(label))  # saca extension (.pg, .txt, etc)
    match = re.search(r"(\d+)", base)

    if match:
        numero = int(match.group())
        prefijo = base[: match.start()].strip("_- ")
        prefijo_legible = ALIAS_PREFIJOS.get(prefijo.lower(), prefijo.replace("_", " ").title() or "Dataset")
        if numero in TAMANOS_REALES_MB:
            nombre = f"{prefijo_legible}-{TAMANOS_REALES_MB[numero]:g}MB"
        else:
            nombre = f"{prefijo_legible}-{formatear_cantidad(numero)}"
        orden = (0, numero, nombre)
    else:
        nombre = base.replace("_", " ").title() or str(label)
        orden = (1, 0, nombre)

    return nombre, orden


df = df_raw.copy()

# Las columnas de tamano de input no aportan informacion util -> se descartan si existen.
for columna in ("input", "input_size"):
    if columna in df.columns:
        df = df.drop(columns=[columna])


df["structure"] = df["explanation"].apply(lambda x: detectar_categoria(x, PALABRAS_ESTRUCTURA))
df["experiment_type"] = df["explanation"].apply(lambda x: detectar_categoria(x, PALABRAS_EXPERIMENTO))

DIVISOR_GRADOS = 5000
DIVISOR_VECINOS = 2500

columnas_tiempo = ["avg_t", "min_t", "max_t", "std_t"]

_nombres_ordenes = df["label"].apply(nombre_y_orden_dataset)
df["dataset"] = _nombres_ordenes.apply(lambda t: t[0])
df["dataset_orden"] = _nombres_ordenes.apply(lambda t: t[1])

ORDEN_POR_DATASET = df.drop_duplicates("dataset").set_index("dataset")["dataset_orden"].to_dict()

TAMANO_REAL_POR_DATASET = {
    nombre: TAMANOS_REALES_MB[orden[1]]
    for nombre, orden in ORDEN_POR_DATASET.items()
    if orden[1] in TAMANOS_REALES_MB
}

def ordenar_datasets(nombres):
    return sorted(set(nombres), key=lambda n: ORDEN_POR_DATASET.get(n, (1, 0, n)))


sin_estructura = df[df["structure"] == "Otro"]["explanation"].unique()
sin_experimento = df[df["experiment_type"] == "Otro"]["explanation"].unique()

print(f"Filas totales: {len(df)}")
print(f"Estructuras detectadas: {df['structure'].unique().tolist()}")
print(f"Tipos de experimento detectados: {df['experiment_type'].unique().tolist()}")
print(f"Datasets detectados (orden): {ordenar_datasets(df['dataset'].unique())}")

if len(sin_estructura) > 0:
    print("\nATENCION - explanations sin estructura reconocida (ajusta PALABRAS_ESTRUCTURA):")
    print(list(sin_estructura))

if len(sin_experimento) > 0:
    print("\nATENCION - explanations sin tipo de experimento reconocido (ajusta PALABRAS_EXPERIMENTO):")
    print(list(sin_experimento))

df.head()


In [ ]:
FIGURES_DIR = Path("figuras")
FIGURES_DIR.mkdir(exist_ok=True)

def guardar(fig, nombre: str):
    ruta = FIGURES_DIR / f"{nombre}.png"
    fig.savefig(ruta, bbox_inches="tight", dpi=150)
    print(f"Guardado: {ruta}")


def _formato_numero(valor, pos=None):
    if valor <= 0:
        return "0"
    if valor >= 1:
        return f"{valor:,.0f}"
    return f"{valor:.3g}"


def aplicar_escala_log(ax, eje: str, limites=None):
    """Activa escala log en 'x' o 'y'. 'limites' (min, max) fija el rango
    en base a los VALORES REALES (evita que barras de error recortadas
    artificialmente para que no crucen el 0 infeccionen el autoscale y
    generen demasiados ticks vacios)."""
    formateador = FuncFormatter(_formato_numero)
    locator = mticker.LogLocator(base=10, numticks=8)
    if eje == "y":
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(formateador)
        ax.yaxis.set_minor_formatter(FuncFormatter(lambda v, p: ""))
        ax.yaxis.set_major_locator(locator)
        if limites:
            ax.set_ylim(limites[0] * 0.5, limites[1] * 2)
    elif eje == "x":
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(formateador)
        ax.xaxis.set_minor_formatter(FuncFormatter(lambda v, p: ""))
        ax.xaxis.set_major_locator(locator)
        if limites:
            ax.set_xlim(limites[0] * 0.5, limites[1] * 2)

## 4. Comparación de tamaños comprimidos

In [ ]:
df_size = (
    df[df["size_in_megabytes"].notna()]
    .groupby(["dataset", "structure"], as_index=False)["size_in_megabytes"]
    .mean()
)

datasets = ordenar_datasets(df_size["dataset"].unique())
estructuras = [e for e in PALETA if e in df_size["structure"].unique()]


incluir_tamano_real = bool(TAMANO_REAL_POR_DATASET) and all(d in TAMANO_REAL_POR_DATASET for d in datasets)
n_grupos = len(estructuras) + (1 if incluir_tamano_real else 0)

x = np.arange(len(datasets))
ancho = 0.8 / max(n_grupos, 1)

fig, ax = plt.subplots(figsize=(max(10, len(datasets) * 1.6), 6))

for i, estructura in enumerate(estructuras):
    sub = df_size[df_size["structure"] == estructura].set_index("dataset").reindex(datasets)
    ax.bar(x + i * ancho, sub["size_in_megabytes"], width=ancho, label=estructura, color=PALETA.get(estructura))

if incluir_tamano_real:
    valores_reales = [TAMANO_REAL_POR_DATASET[d] for d in datasets]
    ax.bar(x + len(estructuras) * ancho, valores_reales, width=ancho, label="Tamaño real (sin comprimir)", color="#495057")

if ESCALA_LOG_TAMANO:
    limite_min = min(df_size["size_in_megabytes"].min(), min(TAMANO_REAL_POR_DATASET.values(), default=df_size["size_in_megabytes"].min()))
    limite_max = max(df_size["size_in_megabytes"].max(), max(TAMANO_REAL_POR_DATASET.values(), default=df_size["size_in_megabytes"].max()))
    aplicar_escala_log(ax, "y", limites=(limite_min, limite_max))

ax.set_xticks(x + ancho * (n_grupos - 1) / 2)
ax.set_xticklabels(datasets, rotation=45, ha="right")
ax.set_ylabel("Tamaño comprimido (MB)")
ax.set_xlabel("Dataset")
ax.set_title("Tamaño comprimido por estructura y dataset")
ax.legend(title="Estructura")
fig.tight_layout()
guardar(fig, "01_tamano_comprimido")
plt.show()

## 5. Ahorro de espacio respecto al tamaño real del grafo

In [ ]:
if TAMANO_REAL_POR_DATASET:
    datasets_local = [d for d in ordenar_datasets(df_size["dataset"].unique()) if d in TAMANO_REAL_POR_DATASET]

    x = np.arange(len(datasets_local))
    ancho = 0.8 / max(len(estructuras), 1)

    fig, ax = plt.subplots(figsize=(max(10, len(datasets_local) * 1.6), 6))
    for i, estructura in enumerate(estructuras):
        sub = df_size[df_size["structure"] == estructura].set_index("dataset").reindex(datasets_local)
        tamano_real = pd.Series({d: TAMANO_REAL_POR_DATASET[d] for d in datasets_local})
        ahorro_pct = (1 - sub["size_in_megabytes"] / tamano_real) * 100
        ax.bar(x + i * ancho, ahorro_pct, width=ancho, label=estructura, color=PALETA.get(estructura))

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x + ancho * (len(estructuras) - 1) / 2)
    ax.set_xticklabels(datasets_local, rotation=45, ha="right")
    ax.set_ylabel("Ahorro de espacio vs. tamaño real del grafo (%)")
    ax.set_xlabel("Dataset")
    ax.set_title("Reducción porcentual de tamaño respecto al grafo real sin comprimir\n(positivo = ocupa menos espacio que el archivo original)")
    ax.legend(title="Estructura")
    fig.tight_layout()
    guardar(fig, "02_ahorro_porcentual")
    plt.show()
else:
    print("No hay tamaños reales configurados en TAMANOS_REALES_MB para los datasets actuales.")


### 5.1 Comparación adicional respecto a la lista de adyacencia

In [ ]:
pivot_size = df_size.pivot(index="dataset", columns="structure", values="size_in_megabytes")

if BASELINE_ESTRUCTURA in pivot_size.columns:
    estructuras_compactas = [e for e in estructuras if e != BASELINE_ESTRUCTURA]
    datasets_local = ordenar_datasets(pivot_size.index)
    pivot_local = pivot_size.reindex(datasets_local)

    x = np.arange(len(datasets_local))
    ancho = 0.8 / max(len(estructuras_compactas), 1)

    fig, ax = plt.subplots(figsize=(max(10, len(datasets_local) * 1.6), 6))
    for i, estructura in enumerate(estructuras_compactas):
        comparables = pivot_local[[estructura, BASELINE_ESTRUCTURA]].dropna()
        ahorro_pct = (1 - comparables[estructura] / comparables[BASELINE_ESTRUCTURA]) * 100
        ahorro_pct = ahorro_pct.reindex(datasets_local)
        ax.bar(x + i * ancho, ahorro_pct, width=ancho, label=estructura, color=PALETA.get(estructura))

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x + ancho * (len(estructuras_compactas) - 1) / 2)
    ax.set_xticklabels(datasets_local, rotation=45, ha="right")
    ax.set_ylabel(f"Ahorro de espacio vs. {BASELINE_ESTRUCTURA} (%)")
    ax.set_xlabel("Dataset")
    ax.set_title(f"Reducción porcentual de tamaño respecto a {BASELINE_ESTRUCTURA}\n(positivo = ocupa menos espacio)")
    ax.legend(title="Estructura")
    fig.tight_layout()
    guardar(fig, "02b_ahorro_vs_adylist")
    plt.show()
else:
    print(f"BASELINE_ESTRUCTURA='{BASELINE_ESTRUCTURA}' no está entre los datos disponibles: {list(pivot_size.columns)}")


## 6. Mapa de calor de tamaños por dataset y estructura

In [ ]:
fig, ax = plt.subplots(figsize=(max(8, len(estructuras) * 2.5), max(6, len(datasets) * 0.45)))
datos_heatmap = pivot_size.reindex(index=datasets, columns=estructuras)
im = ax.imshow(datos_heatmap.values, cmap="YlOrRd", aspect="auto")

ax.set_xticks(range(len(datos_heatmap.columns)))
ax.set_xticklabels(datos_heatmap.columns)
ax.set_yticks(range(len(datos_heatmap.index)))
ax.set_yticklabels(datos_heatmap.index)

valores = datos_heatmap.values
valor_medio = np.nanmin(valores) + (np.nanmax(valores) - np.nanmin(valores)) / 2
for i in range(valores.shape[0]):
    for j in range(valores.shape[1]):
        valor = valores[i, j]
        if not np.isnan(valor):
            # texto blanco sobre celdas oscuras, negro sobre celdas claras, para que siempre se lea
            color_texto = "white" if valor > valor_medio else "black"
            ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=8, color=color_texto)

ax.set_title("Tamaño comprimido (MB) por dataset y estructura")
fig.colorbar(im, ax=ax, label="MB")
fig.tight_layout()
guardar(fig, "03_heatmap_tamanos")
plt.show()

## 7. Tiempos de ejecución

Las consultas de **grado** se corrieron con 1000 repeticiones y las de
**vecinos** con 500 (son más costosas), por lo que se muestran en gráficos
separados en vez de mezclarse en el mismo eje. Por defecto se usa escala
logarítmica en el eje de tiempo, porque la lista de adyacencia es varios
órdenes de magnitud más rápida que k2tree y PEMB.


In [ ]:
def graficar_tiempos(df, tipo_experimento: str, repeticiones: int, nombre_archivo: str):
    sub = df[df["experiment_type"] == tipo_experimento]
    if sub.empty:
        print(f"No hay datos para el tipo de experimento '{tipo_experimento}'.")
        return

    resumen = sub.groupby(["dataset", "structure"], as_index=False).agg(
        avg_t=("avg_t", "mean"),
        std_t=("std_t", "mean"),
    )

    datasets_local = ordenar_datasets(resumen["dataset"].unique())
    estructuras_local = [e for e in PALETA if e in resumen["structure"].unique()]
    x = np.arange(len(datasets_local))
    ancho = 0.8 / max(len(estructuras_local), 1)

    fig, ax = plt.subplots(figsize=(max(10, len(datasets_local) * 1.6), 6))
    for i, estructura in enumerate(estructuras_local):
        s = resumen[resumen["structure"] == estructura].set_index("dataset").reindex(datasets_local)
        yerr_inf = np.minimum(s["std_t"], s["avg_t"] * 0.999) if ESCALA_LOG_TIEMPO else s["std_t"]
        ax.bar(
            x + i * ancho, s["avg_t"], width=ancho, yerr=[yerr_inf, s["std_t"]],
            capsize=3, label=estructura, color=PALETA.get(estructura),
        )

    if ESCALA_LOG_TIEMPO:
        aplicar_escala_log(ax, "y", limites=(resumen["avg_t"].min(), (resumen["avg_t"] + resumen["std_t"]).max()))

    ax.set_xticks(x + ancho * (len(estructuras_local) - 1) / 2)
    ax.set_xticklabels(datasets_local, rotation=45, ha="right")
    ax.set_ylabel(f"Tiempo promedio ({UNIDAD_TIEMPO})")
    ax.set_xlabel("Dataset")
    ax.set_title(
        f"Tiempo de ejecución - consultas de {tipo_experimento.lower()} (n={repeticiones} repeticiones)\n"
        "Barras de error = desviación estándar"
    )
    ax.legend(title="Estructura")
    fig.tight_layout()
    guardar(fig, nombre_archivo)
    plt.show()

### 7.1 Consultas de grado

In [ ]:
graficar_tiempos(df, "Grados", REPS_GRADO, "04_tiempos_grados")

### 7.2 Consultas de vecinos

In [ ]:
graficar_tiempos(df, "Vecinos", REPS_VECINOS, "05_tiempos_vecinos")

## 8. Rango de tiempos observados (mínimo - promedio - máximo)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, (tipo, reps) in zip(axes, [("Grados", REPS_GRADO), ("Vecinos", REPS_VECINOS)]):
    sub = df[df["experiment_type"] == tipo]
    if sub.empty:
        ax.set_title(f"{tipo}: sin datos")
        continue
    resumen = sub.groupby(["dataset", "structure"], as_index=False).agg(
        avg_t=("avg_t", "mean"), min_t=("min_t", "mean"), max_t=("max_t", "mean"),
    )
    datasets_local = ordenar_datasets(resumen["dataset"].unique())
    for estructura in PALETA:
        s = resumen[resumen["structure"] == estructura].set_index("dataset").reindex(datasets_local).dropna()
        if s.empty:
            continue
        # min_t puede venir en 0 (ej. lista de adyacencia); en escala log eso
        # equivale a -infinito, asi que se recorta el error inferior para
        # que nunca toque 0.
        yerr_inf = s["avg_t"] - s["min_t"]
        if ESCALA_LOG_TIEMPO:
            yerr_inf = np.minimum(yerr_inf, s["avg_t"] * 0.999)
        ax.errorbar(
            s.index, s["avg_t"],
            yerr=[yerr_inf, s["max_t"] - s["avg_t"]],
            fmt="o", capsize=4, label=estructura, color=PALETA.get(estructura),
        )
    if ESCALA_LOG_TIEMPO:
        aplicar_escala_log(ax, "y", limites=(resumen["avg_t"].min(), resumen["max_t"].max()))
    ax.set_title(f"Consultas de {tipo.lower()} (n={reps})")
    ax.set_ylabel(f"Tiempo ({UNIDAD_TIEMPO})")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="Estructura")

fig.suptitle("Rango de tiempos observados (mínimo - promedio - máximo)")
fig.tight_layout()
guardar(fig, "06_rango_tiempos")
plt.show()


## 9. Relación entre tamaño y tiempo de consulta

In [ ]:
from matplotlib.lines import Line2D

df_tiempo_prom = (
    df[df["experiment_type"].isin(["Grados", "Vecinos"])]
    .groupby(["dataset", "structure"], as_index=False)["avg_t"].mean()
)
df_tradeoff = df_tiempo_prom.merge(df_size, on=["dataset", "structure"], suffixes=("_tiempo", "_size"))

fig, ax = plt.subplots(figsize=(9, 7))

MARCADORES = ["o", "s", "^", "D", "v", "P", "X"]
datasets_ordenados = ordenar_datasets(df_tradeoff["dataset"].unique())
marcador_por_dataset = {d: MARCADORES[i % len(MARCADORES)] for i, d in enumerate(datasets_ordenados)}
etiqueta_corta_por_dataset = {d: f"Planar {formatear_cantidad(ORDEN_POR_DATASET[d][1])}" for d in datasets_ordenados}


JITTER_X = 0.018

rango_x = df_tradeoff["size_in_megabytes"].max() - df_tradeoff["size_in_megabytes"].min()

for estructura in PALETA:
    s = df_tradeoff[df_tradeoff["structure"] == estructura].copy()
    if s.empty:
        continue
    s["_orden"] = s["dataset"].map(ORDEN_POR_DATASET)
    s = s.sort_values("_orden").reset_index(drop=True)

    ax.plot(s["size_in_megabytes"], s["avg_t"], linewidth=1.5, alpha=0.7, color=PALETA.get(estructura), zorder=1)

    n = len(s)
    for i, (_, fila) in enumerate(s.iterrows()):
        despl = (i - (n - 1) / 2) * JITTER_X * rango_x if n > 1 else 0
        ax.scatter(
            fila["size_in_megabytes"] + despl, fila["avg_t"],
            marker=marcador_por_dataset[fila["dataset"]],
            s=190, color=PALETA.get(estructura), edgecolor="white", linewidth=0.8, zorder=3,
        )

if ESCALA_LOG_TIEMPO:
    ax.set_yscale("log", base=100)
    ax.yaxis.set_major_formatter(FuncFormatter(_formato_numero))
    ax.yaxis.set_minor_formatter(FuncFormatter(lambda v, p: ""))
    ax.yaxis.set_major_locator(mticker.LogLocator(base=100, numticks=8))
    lim = (df_tradeoff["avg_t"].min(), df_tradeoff["avg_t"].max())
    ax.set_ylim(lim[0] * 0.5, lim[1] * 2)
if ESCALA_LOG_TAMANO:
    ax.set_xscale("log", base=100)
    ax.xaxis.set_major_formatter(FuncFormatter(_formato_numero))
    ax.xaxis.set_minor_formatter(FuncFormatter(lambda v, p: ""))
    ax.xaxis.set_major_locator(mticker.LogLocator(base=100, numticks=8))
    lim = (df_tradeoff["size_in_megabytes"].min(), df_tradeoff["size_in_megabytes"].max())
    ax.set_xlim(lim[0] * 0.5, lim[1] * 2)

ax.set_xlabel("Tamaño comprimido (MB)")
ax.set_ylabel(f"Tiempo promedio de consulta ({UNIDAD_TIEMPO})")
ax.set_title("Relación tamaño vs. tiempo de consulta por dataset\n(promedio entre grados y vecinos)")

leyenda_estructura = ax.legend(
    handles=[Line2D([0], [0], color=PALETA[e], linewidth=3, label=e) for e in PALETA if e in df_tradeoff["structure"].unique()],
    title="Estructura", loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0,
)
ax.add_artist(leyenda_estructura)
leyenda_dataset = ax.legend(
    handles=[
        Line2D([0], [0], marker=marcador_por_dataset[d], color="gray", linestyle="None", markersize=10, label=etiqueta_corta_por_dataset[d])
        for d in datasets_ordenados
    ],
    title="Dataset", loc="upper left", bbox_to_anchor=(1.02, 0.55), borderaxespad=0.0,
)

fig.subplots_adjust(right=0.65)
guardar(fig, "07_tradeoff_tamano_tiempo")
plt.show()

## 10. Tabla resumen

In [ ]:
tabla_resumen = df.groupby(["structure", "experiment_type"], as_index=False).agg(
    avg_t_prom=("avg_t", "mean"),
    tamano_prom_mb=("size_in_megabytes", "mean"),
    n_filas=("dataset", "count"),
).sort_values(["experiment_type", "structure"])

tabla_resumen
